In [11]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# 数据增强：随机翻转+归一化，能让模型更稳
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

train_ds = datasets.CIFAR10(root='../data', train=True,  download=False, transform=transform)
test_ds  = datasets.CIFAR10(root='../data', train=False, download=False, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256)
print(f"训练: {len(train_ds)}, 测试: {len(test_ds)}")

训练: 50000, 测试: 10000


In [12]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 特征提取部分：两组"卷积+激活+池化"
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)   # 3通道→16通道
        self.conv2 = nn.Conv2d(16, 64, kernel_size=3, padding=1)  # 16通道→32通道
        self.pool  = nn.MaxPool2d(2)                               # 尺寸减半
        # 分类部分：最后接一个全连接(和您熟悉的MLP一样!)
        self.fc = nn.Linear(64 * 8 * 8, 10)

    def forward(self, x):                    # 输入 [64, 3, 32, 32]
        x = torch.relu(self.conv1(x))        # -> [64,16,32,32]
        x = self.pool(x)                     # -> [64,16,16,16]
        x = torch.relu(self.conv2(x))        # -> [64,32,16,16]
        x = self.pool(x)                     # -> [64,32,8,8]
        x = x.view(-1, 64*8*8)               # 展平 -> [64, 2048]
        return self.fc(x)                    # -> [64, 10]

model = CNN()
print(model)

CNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=4096, out_features=10, bias=True)
)


In [13]:
images, labels = next(iter(train_loader))
print("图像:", images.shape)   # [64, 3, 32, 32] = 批次,通道,高,宽
x = images
for name, layer in [("conv1", model.conv1), ("pool", model.pool)]:
    x = layer(x)
    print(f"经过 {name} 后:", x.shape)

图像: torch.Size([64, 3, 32, 32])
经过 conv1 后: torch.Size([64, 16, 32, 32])
经过 pool 后: torch.Size([64, 16, 16, 16])


In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch():
    model.train()
    total, correct, loss_sum = 0, 0, 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return loss_sum/len(train_loader), correct/total

for epoch in range(15):
    loss, acc = train_one_epoch()
    print(f"Epoch {epoch+1}: loss={loss:.4f}, 训练准确率={acc*100:.2f}%")

Epoch 1: loss=0.6975, 训练准确率=76.19%
Epoch 2: loss=0.6773, 训练准确率=76.82%
Epoch 3: loss=0.6599, 训练准确率=77.44%
Epoch 4: loss=0.6483, 训练准确率=77.80%
Epoch 5: loss=0.6239, 训练准确率=78.47%
Epoch 6: loss=0.6163, 训练准确率=78.76%
Epoch 7: loss=0.6040, 训练准确率=78.97%
Epoch 8: loss=0.5947, 训练准确率=79.46%
Epoch 9: loss=0.5781, 训练准确率=80.09%
Epoch 10: loss=0.5722, 训练准确率=80.35%
Epoch 11: loss=0.5653, 训练准确率=80.50%
Epoch 12: loss=0.5560, 训练准确率=80.74%
Epoch 13: loss=0.5429, 训练准确率=81.34%
Epoch 14: loss=0.5365, 训练准确率=81.36%
Epoch 15: loss=0.5346, 训练准确率=81.28%


In [17]:
def evaluate():
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct/total

print(f"测试集准确率: {evaluate()*100:.2f}%")

测试集准确率: 73.38%
